# 📕 Book 01 — Từ Vi Khuẩn Biển Đến Bản Thảo ISI
### Quy trình nghiên cứu hoạt chất kháng ung thư có trợ lý AI

---

## Cuốn sổ tay này dạy bạn điều gì?

Đây là bản mở rộng đầy đủ của [`quickstart_research_demo.py`](quickstart_research_demo.py).
Chúng ta sẽ đi trọn một **vòng đời nghiên cứu** của một hợp chất thứ cấp
(*secondary metabolite*) phân lập từ vi khuẩn, từ lúc chỉ có một chủng xạ khuẩn
trong đĩa thạch cho tới lúc có file `.docx` gửi tạp chí Q1.

| Chương | Nội dung | Cần Internet? |
|---|---|---|
| 1 | Thiết lập môi trường & khởi tạo trợ lý | Không |
| 2 | Khai phá tài liệu PubMed, thống kê xu hướng | **Có** |
| 3 | Nạp PDF nội bộ phòng lab vào kho tri thức | Không |
| 4 | Truy vấn RAG Hybrid (Vector + BM25 + RRF) | Không |
| 5 | Hồ sơ hóa dược PubChem & Lipinski/Veber | **Có** |
| 6 | Phân tích số liệu hoạt tính IC₅₀ phòng lab | Không |
| 7 | Soạn thảo bản thảo & xuất Microsoft Word | Không |
| 8 | Bài tập tự luyện & hướng mở rộng | — |

> **Cách dùng:** chạy tuần tự từ trên xuống. Các chương cần Internet đều có
> cơ chế *fallback* — nếu mạng hỏng, notebook vẫn chạy tiếp được bằng dữ liệu mẫu.

---

## Bối cảnh khoa học: tại sao lại là vi khuẩn?

Hơn **60%** thuốc chống ung thư đang lưu hành có nguồn gốc hoặc lấy cảm hứng từ
hợp chất tự nhiên. Vi sinh vật — đặc biệt là xạ khuẩn (*Actinomycetes*) biển —
là mỏ vàng vì chúng sống trong môi trường khắc nghiệt (áp suất cao, thiếu dinh dưỡng,
cạnh tranh sinh học dữ dội) nên buộc phải tổng hợp các khung cấu trúc hóa học
phức tạp mà phòng thí nghiệm khó nghĩ ra.

Vài ví dụ kinh điển:

- **Staurosporine** — từ *Streptomyces staurosporeus*, ức chế protein kinase phổ rộng.
- **Salinosporamide A** — từ *Salinispora tropica* (trầm tích biển), ức chế proteasome, đã vào thử nghiệm lâm sàng.
- **Doxorubicin** — từ *Streptomyces peucetius*, xen kẽ DNA, là thuốc nền tảng cho ung thư vú.

### Bốn cơ chế phân tử ta sẽ liên tục gặp lại

1. **Apoptosis (chết theo chương trình)** — con đường nội sinh qua ty thể: hợp chất làm
   mất điện thế màng ty thể (MMP), giải phóng cytochrome c, kích hoạt chuỗi caspase.
2. **Chuỗi Caspase** — caspase-9 (khởi phát) → caspase-3/7 (thi hành) → cắt PARP.
   Phát hiện *cleaved caspase-3* trên Western blot là bằng chứng vàng của apoptosis.
3. **ROS (gốc oxy hóa hoạt động)** — tế bào ung thư vốn đã stress oxy hóa nền cao;
   đẩy ROS vượt ngưỡng sẽ giết chúng chọn lọc hơn tế bào thường.
4. **Chặn chu kỳ tế bào** — dừng ở G2/M (thường do can thiệp vi ống) hoặc G0/G1
   (do ức chế CDK), khiến tế bào không phân chia được.

### Đọc hiểu hai con số quan trọng nhất

- **IC₅₀** — nồng độ ức chế 50% tế bào. *Càng nhỏ càng mạnh.* Dưới 10 μM với dịch
  chiết thô là đáng chú ý; dưới 1 μM với hợp chất tinh khiết là rất mạnh.
- **Selectivity Index (SI)** = IC₅₀ (tế bào thường) / IC₅₀ (tế bào ung thư).
  *Càng lớn càng an toàn.* **SI < 2** nghĩa là chất độc chung chung, không phải thuốc.
  SI > 10 mới đáng theo đuổi.

---

# Chương 1 — Thiết lập môi trường

Notebook nằm trong `books/`, còn mã nguồn nằm ở `app/` cấp trên. Ô đầu tiên
chèn thư mục gốc dự án vào `sys.path` để `import app...` hoạt động, bất kể bạn
mở notebook từ đâu.

In [ ]:
import sys
from pathlib import Path

# Xác định thư mục gốc dự án dù notebook chạy từ books/ hay từ gốc
CWD = Path.cwd()
ROOT_DIR = CWD.parent if CWD.name == 'books' else CWD
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print('Thư mục gốc dự án :', ROOT_DIR)
print('Tồn tại app/      :', (ROOT_DIR / 'app').is_dir())

### Khởi tạo trợ lý nghiên cứu

`ScientificResearchAssistant` là lớp điều phối duy nhất bạn cần nhớ. Nó gói sẵn
6 thành phần con:

| Thuộc tính | Vai trò |
|---|---|
| `.pubmed` | Gọi NCBI E-Utilities tìm bài báo |
| `.europe_pmc` | Tìm và tải PDF Open Access |
| `.pdf_parser` | Bóc tách PDF bằng `pypdfium2` |
| `.pubchem` | Tra cứu cấu trúc & thông số hóa lý |
| `.vector_store` | ChromaDB — bộ nhớ ngữ nghĩa |
| `.retriever` | Hybrid Vector + BM25, hợp nhất bằng RRF |

> ⏳ Lần chạy đầu tiên sẽ hơi lâu vì ChromaDB phải tải mô hình embedding về máy.

In [ ]:
from app.agent.assistant import ScientificResearchAssistant
from app.config import DATA_DIR, RAW_PAPERS_DIR, OUTPUT_DIR, LAB_COMPOUNDS_CSV

assistant = ScientificResearchAssistant()

print('✓ Trợ lý đã sẵn sàng')
print(f'  Kho PDF thô     : {RAW_PAPERS_DIR}')
print(f'  Thư mục kết quả : {OUTPUT_DIR}')
print(f'  Số chunk trong kho RAG hiện có: {assistant.vector_store.count()}')

### Thư viện phụ trợ (tùy chọn)

`pandas` và `matplotlib` **không** nằm trong `requirements.txt`. Notebook được viết
để chạy được cả khi thiếu chúng — chỉ là bảng biểu sẽ hiển thị dạng văn bản thuần.
Muốn có biểu đồ đẹp thì cài thêm:

```bash
pip install pandas matplotlib
```

In [ ]:
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

print(f'pandas     : {"✓ có" if HAS_PANDAS else "✗ thiếu (sẽ in bảng dạng text)"}')
print(f'matplotlib : {"✓ có" if HAS_PLT else "✗ thiếu (sẽ bỏ qua biểu đồ)"}')

---
# Chương 2 — Khai phá tài liệu từ PubMed

## 2.1. Nghệ thuật đặt truy vấn

PubMed không phải Google. Truy vấn tốt là truy vấn **có cấu trúc**, ghép ba trục:

```
[NGUỒN SINH HỌC]  AND  [ĐÍCH UNG THƯ]  AND  [CƠ CHẾ]
   Streptomyces           MCF-7             apoptosis
```

Vài mẹo cú pháp đáng nhớ:

| Cú pháp | Ý nghĩa | Ví dụ |
|---|---|---|
| `"..."` | Khớp đúng cụm từ | `"marine sediment"` |
| `[Title/Abstract]` | Chỉ tìm trong tiêu đề + tóm tắt | `apoptosis[Title/Abstract]` |
| `[MeSH Terms]` | Dùng từ khóa chuẩn hóa y sinh | `Antineoplastic Agents[MeSH Terms]` |
| `AND` / `OR` / `NOT` | Toán tử Boolean (viết HOA) | `MCF-7 OR HeLa` |
| `2020:2026[dp]` | Lọc theo năm xuất bản | `2020:2026[dp]` |

## 2.2. Điều gì xảy ra khi gọi `search_and_index_pubmed`?

```
esearch  →  danh sách PMID
   ↓
efetch   →  XML đầy đủ (tiêu đề, abstract, tác giả, DOI)
   ↓
splitter →  cắt thành chunk ngữ nghĩa, gắn metadata
   ↓
ChromaDB + BM25  →  nạp vào kho, sẵn sàng truy vấn
```

Nghĩa là một lệnh vừa **tìm** vừa **học** — bài báo lấy về lập tức trở thành
tri thức mà RAG truy xuất được ở Chương 4.

In [ ]:
QUERIES = {
    'Xạ khuẩn biển & apoptosis':
        'Streptomyces marine apoptosis anticancer[Title/Abstract]',
    'Bacillus & dòng tế bào ung thư vú':
        'Bacillus secondary metabolite MCF-7 cytotoxicity',
    'Kích hoạt Caspase-3 bởi hợp chất vi khuẩn':
        'bacterial metabolite caspase-3 activation cancer cell line',
}

collected = {}

for label, q in QUERIES.items():
    print(f'\n{"=" * 70}\n▶ {label}\n  Truy vấn: {q}\n{"=" * 70}')
    try:
        articles = assistant.search_and_index_pubmed(query=q, max_results=5)
    except Exception as e:
        print(f'  ⚠ Không gọi được PubMed ({type(e).__name__}: {e})')
        print('    → Bỏ qua, các chương sau vẫn chạy được.')
        articles = []

    collected[label] = articles
    for a in articles:
        print(f'  [{a.pub_year}] {a.title[:80]}')
        print(f'         {a.journal}  |  PMID {a.pmid}')

total = sum(len(v) for v in collected.values())
print(f'\n✓ Thu được {total} bài báo. Kho RAG hiện có {assistant.vector_store.count()} chunk.')

## 2.3. Đọc xu hướng: bao nhiêu bài mỗi năm?

Phân bố bài báo theo năm cho bạn biết chủ đề đang **nóng lên** hay **nguội đi** —
một dữ kiện rất hữu ích để viết câu mở đầu phần Introduction kiểu
*"Research interest in marine actinomycetes has grown steadily since ..."*.

In [ ]:
from collections import Counter

all_articles = [a for arts in collected.values() for a in arts]

if not all_articles:
    print('Chưa có bài báo nào (có thể do mất mạng ở ô trên). Bỏ qua chương này.')
else:
    year_counts = Counter(a.pub_year for a in all_articles if a.pub_year != 'N/A')
    journal_counts = Counter(a.journal for a in all_articles)

    print('── Số bài theo năm ──')
    for year, n in sorted(year_counts.items()):
        print(f'  {year}  {"█" * n}  ({n})')

    print('\n── Tạp chí xuất hiện nhiều nhất ──')
    for jr, n in journal_counts.most_common(5):
        print(f'  {n:>2}×  {jr}')

    if HAS_PLT and year_counts:
        years = sorted(year_counts)
        plt.figure(figsize=(8, 3.5))
        plt.bar(years, [year_counts[y] for y in years], color='#2a7f62')
        plt.title('Phân bố bài báo theo năm xuất bản')
        plt.xlabel('Năm'); plt.ylabel('Số bài')
        plt.tight_layout(); plt.show()

---
# Chương 3 — Nạp PDF nội bộ của phòng lab

Abstract trên PubMed chỉ là phần nổi. Phần thân bài — nơi ghi **quy trình nuôi cấy**,
**điều kiện lên men**, **bảng IC₅₀ chi tiết** — chỉ có trong file PDF đầy đủ.

`ingest_pdf_file()` làm ba việc:

1. Bóc văn bản từng trang bằng `pypdfium2`, nhận diện ranh giới các mục
   (Introduction / Methods / Results / Discussion).
2. **Nhận diện thực thể bằng regex**: tên chủng vi khuẩn, dòng tế bào ung thư,
   con đường tín hiệu, và các mẩu câu có chứa giá trị IC₅₀.
3. Cắt chunk có chồng lấn (*overlap*) rồi nạp vào ChromaDB + BM25.

> 📁 Đặt file PDF của bạn vào `data/raw_papers/` rồi chạy ô dưới.

In [ ]:
pdf_files = sorted(RAW_PAPERS_DIR.glob('*.pdf'))

if not pdf_files:
    print(f'Chưa có PDF nào trong {RAW_PAPERS_DIR}')
    print('→ Thả vài file .pdf vào đó rồi chạy lại ô này.')
else:
    print(f'Tìm thấy {len(pdf_files)} file PDF. Bắt đầu nạp...\n')
    for pdf in pdf_files[:3]:          # giới hạn 3 file cho lần chạy demo
        try:
            info = assistant.ingest_pdf_file(pdf)
        except Exception as e:
            print(f'⚠ Lỗi khi đọc {pdf.name}: {e}')
            continue

        print(f"📄 {info['file_name']}")
        print(f"   Tiêu đề    : {info['title'][:70]}")
        print(f"   Số trang   : {info['pages']}  |  Chunk nạp: {info['chunks_indexed']}")
        print(f"   Chủng      : {', '.join(info['strains']) or '—'}")
        print(f"   Dòng tế bào: {', '.join(info['cell_lines']) or '—'}")
        for snip in info['ic50_snippets'][:2]:
            print(f"   ↳ IC50: {snip[:100]}")
        print()

print(f'Tổng chunk trong kho RAG: {assistant.vector_store.count()}')

---
# Chương 4 — Truy vấn kho tri thức bằng RAG Hybrid

## 4.1. Vì sao cần *hybrid*, không dùng vector thuần?

Tìm kiếm ngữ nghĩa (dense vector) rất giỏi hiểu **ý**, nhưng lại tệ ở đúng chỗ
mà sinh học cần nhất: **danh pháp chính xác**.

| Tình huống | Vector thuần | BM25 (từ khóa) |
|---|---|---|
| "cơ chế chết tế bào theo chương trình" | ✓ tìm ra bài nói về apoptosis | ✗ trượt vì không khớp chữ |
| `Streptomyces sp. VN-08` | ✗ dễ nhầm sang chủng khác | ✓ khớp chính xác |
| `Caspase-3` vs `Caspase-9` | ✗ hai vector gần như trùng nhau | ✓ phân biệt rạch ròi |

Với nghiên cứu vi sinh, nhầm `VN-08` thành `VN-03` là sai hoàn toàn. Nên ta chạy
**cả hai** rồi hợp nhất kết quả.

## 4.2. Reciprocal Rank Fusion (RRF)

Vấn đề: điểm cosine (0–1) và điểm BM25 (0–∞) không cùng thang, cộng thẳng là vô nghĩa.
RRF giải quyết bằng cách **vứt bỏ điểm số, chỉ giữ thứ hạng**:

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}, \quad k = 60$$

Ý nghĩa trực giác: một tài liệu **được cả hai bên cùng gật đầu** sẽ luôn thắng
tài liệu chỉ được một bên xếp hạng nhất. Hằng số `k = 60` làm mượt, tránh để
vị trí #1 áp đảo toàn bộ.

In [ ]:
RAG_QUESTIONS = [
    'caspase-3 cleavage mitochondrial apoptosis pathway',
    'IC50 value MCF-7 breast cancer cytotoxicity assay',
    'reactive oxygen species ROS generation cancer cell death',
    'G2/M cell cycle arrest natural compound',
]

if assistant.vector_store.count() == 0:
    print('⚠ Kho RAG đang rỗng — hãy chạy Chương 2 hoặc Chương 3 trước.')
else:
    for q in RAG_QUESTIONS:
        print(f'\n❓ {q}\n{"-" * 70}')
        hits = assistant.query_rag(q, top_k=3)
        if not hits:
            print('   (không có kết quả)')
        for h in hits:
            meta_h = h.get('metadata', {})
            src = meta_h.get('title') or meta_h.get('source', 'N/A')
            print(f"   #{h['rank']}  RRF={h['rrf_score']:.4f}  ← {src[:55]}")
            print(f"       {h['content'][:150].strip()}...")

## 4.3. Xem tận mắt cách RRF hợp nhất

Ô dưới chạy riêng nhánh vector và nhánh hybrid để bạn so sánh thứ hạng. Hãy chú ý
những tài liệu **nhảy hạng** — đó chính là dấu vết BM25 đã can thiệp.

In [ ]:
probe = 'Streptomyces apoptosis caspase'

if assistant.vector_store.count() == 0:
    print('⚠ Kho RAG rỗng — bỏ qua.')
else:
    vec_hits = assistant.vector_store.search(probe, top_k=5)
    hyb_hits = assistant.retriever.search(probe, top_k=5)

    def head(items, key):
        return [(i + 1, (it.get(key, '') or '')[:45]) for i, it in enumerate(items)]

    print(f'Truy vấn: "{probe}"\n')
    print('── Chỉ Vector (ngữ nghĩa) ──')
    for rank, txt in head(vec_hits, 'content'):
        print(f'  {rank}. {txt}...')

    print('\n── Hybrid Vector + BM25 (RRF) ──')
    for rank, txt in head(hyb_hits, 'content'):
        print(f'  {rank}. {txt}...')

    print('\n💡 Tài liệu đổi vị trí giữa hai danh sách = phần đóng góp của BM25.')

---
# Chương 5 — Hồ sơ hóa dược: PubChem, Lipinski và Veber

## 5.1. Tại sao phải sàng lọc *trước khi* làm thí nghiệm?

Một hợp chất diệt tế bào ung thư trong đĩa petri chưa chắc thành thuốc. Nó còn phải
**vào được cơ thể** — hấp thu qua ruột, sống sót qua gan, đến được khối u.

Christopher Lipinski (Pfizer, 1997) phân tích hàng nghìn thuốc uống thành công
và rút ra **Quy tắc 5**:

| Tiêu chí | Ngưỡng | Vì sao |
|---|---|---|
| Khối lượng phân tử (MW) | ≤ 500 Da | Phân tử quá to không lọt qua màng ruột |
| LogP (ưa dầu/ưa nước) | ≤ 5 | Quá ưa dầu → kẹt trong mỡ, không tan trong máu |
| Cho liên kết H (HBD) | ≤ 5 | Nhiều H cho → bám nước quá chặt, khó qua màng lipid |
| Nhận liên kết H (HBA) | ≤ 10 | Tương tự trên |

Bổ sung **tiêu chuẩn Veber (2002)**: số liên kết xoay được ≤ 10. Phân tử quá
mềm dẻo sẽ tốn năng lượng entropy khi gắn vào đích, làm giảm ái lực.

> ⚠️ **Đừng thần thánh hóa quy tắc này.** Rất nhiều hợp chất tự nhiên kháng ung thư
> mạnh nhất *vi phạm* Lipinski (paclitaxel, rapamycin, các macrolide). Vi phạm không
> có nghĩa là bỏ — nó có nghĩa là cần đường tiêm, hoặc cần hệ dẫn thuốc nano, hoặc
> cần cải biên cấu trúc. Module `druglikeness.py` viết đúng sắc thái này vào
> đoạn văn Discussion tự sinh.

In [ ]:
COMPOUNDS = [
    'Staurosporine',      # Streptomyces staurosporeus — ức chế kinase phổ rộng
    'Doxorubicin',        # Streptomyces peucetius — xen kẽ DNA
    'Salinosporamide A',  # Salinispora tropica — ức chế proteasome
    'Rapamycin',          # Streptomyces hygroscopicus — ức chế mTOR
    'Paclitaxel',         # đối chứng thực vật (Taxus brevifolia)
]

reports = []
for name in COMPOUNDS:
    try:
        rep = assistant.analyze_chemical_compound(name)
    except Exception as e:
        print(f'⚠ {name}: lỗi mạng ({type(e).__name__})')
        continue
    if rep is None:
        print(f'✗ {name}: PubChem không tìm thấy')
        continue
    reports.append(rep)
    flag = '✓ ĐẠT' if rep.is_lipinski_compliant else '✗ VI PHẠM'
    print(f'{flag}  {rep.compound_name}')
    print(f'        MW={rep.molecular_weight:.1f} Da | XLogP={rep.xlogp} | '
          f'HBD={rep.h_bond_donors} | HBA={rep.h_bond_acceptors} | RotB={rep.rotatable_bonds}')
    print(f'        Kết luận: {rep.summary_verdict}\n')

## 5.2. Bảng so sánh — dán thẳng vào bài báo được

Bảng này chính là **Table 2** điển hình trong phần Results của một bài đăng
*Journal of Natural Products* hay *European Journal of Medicinal Chemistry*.

In [ ]:
if not reports:
    print('Chưa có hồ sơ hóa dược nào (kiểm tra kết nối mạng ở ô trên).')
elif HAS_PANDAS:
    df_chem = pd.DataFrame([{
        'Compound': r.compound_name,
        'MW (Da)': round(r.molecular_weight, 2),
        'XLogP': r.xlogp,
        'HBD': r.h_bond_donors,
        'HBA': r.h_bond_acceptors,
        'RotB': r.rotatable_bonds,
        'Lipinski viol.': r.lipinski_violations,
        'Veber viol.': r.veber_violations,
        'Drug-like': 'Yes' if r.is_lipinski_compliant else 'No',
    } for r in reports])
    display(df_chem)
else:
    hdr = f"{'Compound':<22}{'MW':>9}{'XLogP':>8}{'HBD':>5}{'HBA':>5}{'RotB':>6}{'Viol':>6}"
    print(hdr); print('-' * len(hdr))
    for r in reports:
        print(f'{r.compound_name[:21]:<22}{r.molecular_weight:>9.1f}{r.xlogp:>8}'
              f'{r.h_bond_donors:>5}{r.h_bond_acceptors:>5}{r.rotatable_bonds:>6}'
              f'{r.lipinski_violations:>6}')

## 5.3. Bản đồ không gian hóa học

Trục MW × LogP là cách quen thuộc nhất để nhìn một thư viện hợp chất. Vùng chữ nhật
góc dưới-trái là **không gian thuốc uống**; điểm nằm ngoài vùng đó là ứng viên cho
đường tiêm hoặc cho hệ dẫn thuốc.

In [ ]:
if reports and HAS_PLT:
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    for r in reports:
        ok = r.is_lipinski_compliant
        ax.scatter(r.molecular_weight, r.xlogp,
                   s=180, alpha=0.8,
                   c='#2a7f62' if ok else '#c0392b',
                   edgecolors='black', linewidths=0.6)
        ax.annotate(r.compound_name[:16], (r.molecular_weight, r.xlogp),
                    textcoords='offset points', xytext=(8, 6), fontsize=8.5)

    ax.axvline(500, ls='--', c='gray', lw=1)
    ax.axhline(5, ls='--', c='gray', lw=1)
    ax.axvspan(0, 500, ymin=0, ymax=1, alpha=0.05, color='green')
    ax.set_xlabel('Khối lượng phân tử (Da)  →  ngưỡng Lipinski = 500')
    ax.set_ylabel('XLogP  →  ngưỡng Lipinski = 5')
    ax.set_title('Không gian hóa học: xanh = đạt Lipinski, đỏ = vi phạm')
    ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
else:
    print('Bỏ qua biểu đồ (thiếu matplotlib hoặc chưa có dữ liệu).')

## 5.4. Đoạn văn Discussion tự sinh

Đây là thứ tiết kiệm thời gian nhất của module hóa dược: một đoạn văn tiếng Anh học
thuật, đúng văn phong tạp chí, **chèn thẳng vào mục Discussion** được. Lưu ý cách nó
đổi giọng tùy theo hợp chất đạt hay vi phạm Lipinski.

In [ ]:
for r in reports[:3]:
    print(f'▶ {r.compound_name}\n{"─" * 70}')
    print(r.scientific_narrative)
    if r.detailed_violations:
        print(f'\n  Chi tiết vi phạm: {"; ".join(r.detailed_violations)}')
    print()

---
# Chương 6 — Phân tích số liệu hoạt tính của phòng lab

File `data/lab_compounds.csv` mô phỏng đúng định dạng mà một phòng thí nghiệm sinh học
thực tế xuất ra sau các mẻ thử MTT / CCK-8:

| Cột | Ý nghĩa |
|---|---|
| `compound_id` | Mã nội bộ hợp chất |
| `source_strain` | Chủng vi khuẩn phân lập ra nó |
| `cancer_cell_line` | Dòng tế bào ung thư đem thử |
| `ic50_uM` | IC₅₀ tính bằng μM — **càng nhỏ càng mạnh** |
| `assay_method` | Phương pháp & thời gian ủ |
| `selectivity_index` | SI — **càng lớn càng an toàn** |
| `apoptosis_status` | Bằng chứng cơ chế quan sát được |

### Nhận diện các dòng tế bào hay gặp

| Dòng | Mô | Đặc điểm cần biết |
|---|---|---|
| **MCF-7** | Ung thư vú | ER dương tính, *thiếu caspase-3* — nhớ kiểm chứng bằng chỉ dấu khác |
| **HeLa** | Ung thư cổ tử cung | Dòng bất tử đầu tiên, chuẩn mực so sánh |
| **A549** | Ung thư phổi | KRAS đột biến, kháng nhiều thuốc |
| **HepG2** | Ung thư gan | Còn giữ chuyển hóa gan → tốt cho thử độc tính |

In [ ]:
import csv

rows = []
if LAB_COMPOUNDS_CSV.exists():
    with open(LAB_COMPOUNDS_CSV, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            row['ic50_uM'] = float(row['ic50_uM'])
            row['selectivity_index'] = float(row['selectivity_index'])
            rows.append(row)
    print(f'✓ Đọc {len(rows)} dòng từ {LAB_COMPOUNDS_CSV.name}\n')
else:
    print(f'✗ Không thấy {LAB_COMPOUNDS_CSV}')

if rows:
    hdr = f"{'ID':<8}{'Hợp chất':<22}{'Dòng TB':<10}{'IC50 (μM)':>11}{'SI':>7}"
    print(hdr); print('-' * len(hdr))
    for r in sorted(rows, key=lambda x: x['ic50_uM']):
        print(f"{r['compound_id']:<8}{r['compound_name'][:21]:<22}"
              f"{r['cancer_cell_line']:<10}{r['ic50_uM']:>11.2f}{r['selectivity_index']:>7.1f}")

## 6.1. Xếp hạng ứng viên — đọc hai chiều

Đây là chỗ người mới hay mắc lỗi: **chỉ nhìn IC₅₀**. Một hợp chất IC₅₀ = 0.5 μM
nghe rất mạnh, nhưng nếu SI = 1.5 thì nó giết tế bào lành cũng nhanh y hệt — đó là
chất độc, không phải thuốc.

Ứng viên tốt phải **đồng thời** mạnh (IC₅₀ thấp) và chọn lọc (SI cao).

In [ ]:
if rows:
    print('🏆 XẾP THEO ĐỘ MẠNH (IC50 thấp nhất)')
    for r in sorted(rows, key=lambda x: x['ic50_uM'])[:3]:
        print(f"   {r['ic50_uM']:>6.2f} μM  {r['compound_name']} trên {r['cancer_cell_line']}")

    print('\n🛡  XẾP THEO ĐỘ CHỌN LỌC (SI cao nhất)')
    for r in sorted(rows, key=lambda x: -x['selectivity_index'])[:3]:
        print(f"   SI={r['selectivity_index']:>5.1f}  {r['compound_name']} trên {r['cancer_cell_line']}")

    # Điểm tổng hợp: thưởng cho vừa mạnh vừa chọn lọc
    print('\n⭐ ỨNG VIÊN TỐT NHẤT (SI / IC50 — cân bằng cả hai)')
    scored = sorted(rows, key=lambda x: -(x['selectivity_index'] / x['ic50_uM']))
    for r in scored[:3]:
        score = r['selectivity_index'] / r['ic50_uM']
        print(f"   Điểm={score:>6.2f}  {r['compound_name']} ({r['source_strain']})")
        print(f"              → {r['apoptosis_status']}")

In [ ]:
if rows and HAS_PLT:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for r in rows:
        strong = r['ic50_uM'] < 5 and r['selectivity_index'] > 10
        ax.scatter(r['ic50_uM'], r['selectivity_index'],
                   s=170, alpha=0.82,
                   c='#2a7f62' if strong else '#7f8c8d',
                   edgecolors='black', linewidths=0.6)
        ax.annotate(f"{r['compound_name'][:14]}\n{r['cancer_cell_line']}",
                    (r['ic50_uM'], r['selectivity_index']),
                    textcoords='offset points', xytext=(8, -4), fontsize=7.5)

    ax.axvline(5, ls='--', c='gray', lw=1)
    ax.axhline(10, ls='--', c='gray', lw=1)
    ax.set_xlabel('IC₅₀ (μM)  →  càng trái càng mạnh')
    ax.set_ylabel('Selectivity Index  →  càng lên càng an toàn')
    ax.set_title('Góc trên-trái = vùng ứng viên tiềm năng nhất')
    ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
else:
    print('Bỏ qua biểu đồ (thiếu matplotlib hoặc chưa có dữ liệu).')

---
# Chương 7 — Soạn thảo bản thảo và xuất Microsoft Word

## 7.1. Bộ khung một bài IMRaD hóa sinh

| Mục | Nhiệm vụ | Sai lầm thường gặp |
|---|---|---|
| **Title** | Nêu chủng + hoạt tính + đích | Dài dòng, thiếu tên chủng |
| **Abstract** | 200–250 từ, có con số cụ thể | Viết chung chung, không nêu IC₅₀ |
| **1. Introduction** | Từ rộng → hẹp, kết bằng khoảng trống tri thức | Không nói rõ *vì sao* cần nghiên cứu này |
| **2. Results and Discussion** | Trình bày dữ liệu **kèm diễn giải** | Chỉ mô tả bảng, không giải thích |
| **3. Experimental Section** | Đủ chi tiết để người khác lặp lại | Thiếu điều kiện nuôi cấy, thiếu đối chứng |
| **4. Conclusion** | Ý nghĩa + hướng tiếp theo | Lặp lại nguyên xi abstract |
| **References** | Đúng định dạng tạp chí đích | Trích dẫn cũ, thiếu bài 3 năm gần đây |

## 7.2. `PaperMetadata` — phiếu khai báo nghiên cứu

Bạn không viết bài báo, bạn **khai báo sự thật thí nghiệm**. Bộ sinh sẽ lấy các
trường này, ghép với ngữ cảnh RAG (Chương 2–4) và hồ sơ Lipinski (Chương 5) để
dựng bản thảo.

In [ ]:
from app.tools.writer.paper_generator import PaperMetadata

meta = PaperMetadata(
    title='',  # để trống → hệ thống tự sinh tiêu đề theo chuẩn tạp chí
    authors=['Nguyen Thuy Linh', 'Tran Quang Minh', 'Pham Thi Hoa'],
    affiliations=[
        'Department of Marine Biotechnology, Vietnam National University',
        'Institute of Pharmacognosy & Natural Product Chemistry',
    ],
    target_journal='Journal of Natural Products',
    bacterial_strain='Streptomyces sp. VN-08',
    isolation_source='Marine sediment, Nha Trang Bay, Vietnam (depth 45 m)',
    compound_name='Staurosporine',
    target_cancer_cells=['MCF-7', 'HeLa', 'HepG2'],
    measured_ic50='3.45 ± 0.28 μM (MCF-7); 4.12 ± 0.31 μM (HeLa)',
    observed_mechanisms=[
        'Apoptosis induction via the intrinsic mitochondrial pathway',
        'Cleavage of Caspase-3 and PARP confirmed by Western blot',
        'G2/M cell cycle arrest (flow cytometry, PI staining)',
        'Elevation of intracellular ROS (DCFH-DA assay)',
        'Loss of mitochondrial membrane potential (JC-1 assay)',
    ],
)

print('✓ Đã khai báo metadata')
print(f'  Chủng     : {meta.bacterial_strain}')
print(f'  Nguồn     : {meta.isolation_source}')
print(f'  Tạp chí   : {meta.target_journal}')
print(f'  Dòng TB   : {", ".join(meta.target_cancer_cells)}')

## 7.3. Gắn số liệu thực nghiệm thật vào bản thảo

`bioassay_data` nhận một danh sách `dict` — chính là các dòng CSV ở Chương 6.
Chúng sẽ được dựng thành **bảng dữ liệu hoạt tính** trong file Word.

In [ ]:
bioassay = [
    {
        'compound': r['compound_name'],
        'cell_line': r['cancer_cell_line'],
        'ic50': f"{r['ic50_uM']:.2f}",
        'assay': r['assay_method'],
        'selectivity_index': f"{r['selectivity_index']:.1f}",
        'mechanism': r['apoptosis_status'],
    }
    for r in rows
] if rows else None

print(f'Số dòng dữ liệu sinh học đưa vào bản thảo: {len(bioassay) if bioassay else 0}')

## 7.4. Sinh bản thảo

`write_full_paper()` chạy bốn bước:

1. Truy vấn RAG bằng chính metadata → lấy ngữ cảnh + danh sách tài liệu tham khảo.
2. Tra PubChem cho `compound_name` → sinh bảng Lipinski + đoạn văn drug-likeness.
3. Dựng từng mục theo template của tạp chí đích.
4. Xuất `.md` và `.docx` (Times New Roman 12pt, lề 1 inch) vào `outputs/`.

> Nếu có `OPENAI_API_KEY` trong `.env`, bước 3 dùng LLM để trau chuốt văn phong.
> Không có key thì hệ thống chạy **offline hoàn toàn** bằng template — vẫn ra bản thảo đầy đủ.

In [ ]:
manuscript = assistant.write_full_paper(meta=meta, bioassay_data=bioassay)

print('=' * 72)
print('TIÊU ĐỀ:', manuscript.title)
print('=' * 72)
print('\nTỪ KHÓA:', ', '.join(manuscript.keywords))
print('\nABSTRACT:\n')
print(manuscript.abstract)
print(f'\n\nCác mục đã sinh: {list(manuscript.sections.keys())}')
print(f'Số tài liệu tham khảo: {len(manuscript.references)}')
print(f'\n📄 File Word: {manuscript.docx_file_path}')

In [ ]:
# Xem thử phần Results and Discussion
for name, body in manuscript.sections.items():
    if 'result' in name.lower() or 'discussion' in name.lower():
        print(f'▼ {name}\n{"─" * 72}')
        print(body[:2000])
        print('\n[... rút gọn ...]')
        break

In [ ]:
# Liệt kê toàn bộ sản phẩm đã xuất ra thư mục outputs/
products = sorted(OUTPUT_DIR.glob('*'), key=lambda p: p.stat().st_mtime, reverse=True)
print(f'Thư mục: {OUTPUT_DIR}\n')
for p in products[:10]:
    print(f'  {p.stat().st_size / 1024:>8.1f} KB   {p.name}')

---
# Chương 8 — Bài tập tự luyện

### Mức cơ bản

1. **Đổi chủng nghiên cứu.** Sửa `meta.bacterial_strain` sang `Bacillus velezensis BV-12`
   và `compound_name` sang `Bacillopeptin B`, chạy lại Chương 7. Bản thảo thay đổi
   những chỗ nào? Vì sao phần Introduction lại khác?

2. **Thử một truy vấn PubMed của riêng bạn.** Dùng cú pháp `[MeSH Terms]` và lọc năm
   `2022:2026[dp]`. So sánh số lượng kết quả với truy vấn tự do.

3. **Tra một hợp chất vi phạm Lipinski nặng** (gợi ý: `Rapamycin`, `Amphotericin B`).
   Đọc kỹ `scientific_narrative` — hệ thống đề xuất hướng khắc phục nào?

### Mức nâng cao

4. **Đường cong liều–đáp ứng.** Viết hàm khớp phương trình Hill bốn tham số để tính
   IC₅₀ từ số liệu OD thô của máy đọc đĩa:

   $$y = \text{Bottom} + \frac{\text{Top} - \text{Bottom}}{1 + \left(\dfrac{x}{\text{IC}_{50}}\right)^{h}}$$

   (cần `scipy.optimize.curve_fit`). Đây chính là nội dung notebook
   `03_dose_response_ic50_curve.ipynb` mà README gợi ý.

5. **Đo chất lượng RAG.** Tự tay soạn 10 cặp (câu hỏi, đoạn văn đúng), rồi tính
   *Recall@3* cho ba chế độ: chỉ vector, chỉ BM25, và hybrid RRF. Hybrid thắng
   bao nhiêu phần trăm? Với loại câu hỏi nào thì nó **không** thắng?

6. **Sinh bản thảo theo mẻ.** Lặp qua từng `compound_id` trong `lab_compounds.csv`,
   sinh một bản thảo riêng cho mỗi hợp chất, đặt tên file theo mã hợp chất.

7. **Chỉnh `k` của RRF.** Sửa `HybridRetriever(vector_store=..., rrf_k=10)` rồi so
   với mặc định `k=60`. `k` nhỏ khiến top-1 của mỗi nhánh áp đảo hơn — điều đó tốt
   hay xấu cho truy vấn chứa mã chủng như `VN-08`?

---

## Tóm tắt toàn bộ đường ống

```
PubMed / Europe PMC ──┐
                      ├──► Text Splitter ──► ChromaDB + BM25 ──► RRF ──┐
PDF phòng lab ────────┘                                                │
                                                                       ▼
PubChem ──► Lipinski / Veber ──► đoạn văn drug-likeness ──────► Paper Generator
                                                                       ▲
lab_compounds.csv ──► bảng IC₅₀ & Selectivity Index ────────────────────┘
                                                                       │
                                                          outputs/*.docx + *.md
```

## Đọc thêm trong dự án

- [`research_syllabus_and_rag_blueprint.md`](../research_syllabus_and_rag_blueprint.md) — giáo trình & thiết kế kiến trúc
- [`app/tools/rag/hybrid_retriever.py`](../app/tools/rag/hybrid_retriever.py) — cài đặt RRF
- [`app/tools/chemistry/druglikeness.py`](../app/tools/chemistry/druglikeness.py) — logic Lipinski & sinh văn bản
- [`app/tools/writer/templates.py`](../app/tools/writer/templates.py) — khung mẫu từng tạp chí

Chạy giao diện web để thao tác nhanh không cần code: `streamlit run appst.py`